# DKT–MARL — Eksperimen ulang dari awal

DBE-KT22 · DKT berbasis GRU · Evaluasi replay dengan anggaran 20 soal.


## Alur eksperimen

1. Jalankan `kc_smoke` untuk verifikasi.
2. Setelah lolos, ganti ke `kc_audit` dan jalankan dari atas untuk lima seed.
3. Gunakan `RUN_ID` yang sama untuk melanjutkan run yang terputus. Gunakan ID baru untuk pelatihan baru.

Audit memakai split yang sama dengan penelitian sebelumnya untuk keterbandingan. Hasilnya merupakan evaluasi replay historis. Kelompok final test tidak dievaluasi dari notebook ini.


# 1. Persiapan lingkungan dan konfigurasi

## 1.1 Tujuan dan protokol komputasi

Semua model dilatih menggunakan DKT, lingkungan CAT, batas 20 soal, dan reward yang sama. Perbedaannya adalah *policy representation*: RL-Flat memetakan state langsung ke aksi soal; RL-Feat menggunakan state dan fitur soal; MARL memilih strategi, KC, dan soal secara berurutan.

**Catatan tentang penelitian ulang:** data final test penelitian sebelumnya sudah pernah dilihat. Walaupun notebook ini menjaga kelompok test tidak dipakai untuk tuning, kelompok tersebut tidak lagi merupakan holdout yang sepenuhnya baru secara sejarah penelitian. Laporkan keterbatasan ini jika memakai dataset yang sama.

## Penyimpanan

ZIP DBE-KT22 dibaca dari `MyDrive/Riset_MARL/input/`. Checkpoint dan keluaran ditulis ke `MyDrive/Riset_MARL_Rerun/<RUN_ID>/`. Arsip agregat untuk GitHub dibuat setelah evaluasi selesai.


In [ ]:
import os
import re
from pathlib import Path

MODE = 'kc_smoke'  # Ubah ke 'kc_audit' setelah smoke berhasil.
RUN_ID = 'rerun-001'

if MODE not in ('kc_smoke', 'kc_audit'):
    raise ValueError('MODE harus kc_smoke atau kc_audit.')
if not re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9_-]{0,63}', RUN_ID):
    raise ValueError('RUN_ID hanya menerima huruf, angka, tanda hubung, dan garis bawah.')

try:
    from google.colab import drive
except ImportError:
    IN_COLAB = False
    base_dir = Path.cwd()
else:
    IN_COLAB = True
    drive.mount('/content/drive')
    base_dir = Path('/content/drive/MyDrive')

INPUT = Path(os.environ.get('DKT_MARL_INPUT', base_dir / 'Riset_MARL' / 'input'))
ROOT = base_dir / 'Riset_MARL_Rerun'
WORKSPACE = ROOT / RUN_ID
WORKSPACE.mkdir(parents=True, exist_ok=True)
print('Dataset:', INPUT)
print('Output:', WORKSPACE)


In [ ]:
import os, sys, json, zipfile, random, copy, warnings, io, hashlib, shutil, time
from pathlib import Path
from dataclasses import dataclass, asdict
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from sklearn.metrics import roc_auc_score
from scipy.stats import wilcoxon
from scipy.special import expit
from IPython.display import display

# MODE ditetapkan hanya pada sel SETUP di atas; jangan didefinisikan ulang.
SEED = 42
MULTI_SEEDS = [42, 123, 321, 2026, 2027]
ABLATION_SEEDS = [42, 123]
FINAL_TEST_UNLOCK = False  # Ubah True hanya sesudah mode freeze berhasil.
UPLOAD_DARI_LAPTOP = False
assert MODE in ('kc_smoke', 'kc_audit')

# ROOT/INPUT/WORKSPACE telah dibuat di sel SETUP GOOGLE DRIVE.
# Jalur ini terpisah sepenuhnya dari eksperimen terdahulu.
MODE_ROOT = WORKSPACE / 'runs' / MODE
MODE_ROOT.mkdir(parents=True, exist_ok=True)
REPORTS = MODE_ROOT/'reports'
TABLES = MODE_ROOT/'tables'
FIGURES = MODE_ROOT/'figures'
for p in (REPORTS, TABLES, FIGURES): p.mkdir(parents=True, exist_ok=True)
DKT_SCOPE = 'smoke' if MODE in ('smoke', 'kc_smoke') else 'full'
DKT_DIR = WORKSPACE/'dkt'/DKT_SCOPE
DKT_DIR.mkdir(parents=True, exist_ok=True)

@dataclass
class Config:
    seed:int = SEED
    split_seed:int = 20261115
    test_length:int = 20
    target_frac:float = .20
    hidden:int = 48
    emb:int = 32
    dkt_lr:float = .002
    ppo_lr:float = .0003
    gamma:float = .98
    gae_lambda:float = .95
    clip:float = .20
    reward_coverage:float = 1.0
    reward_bce_gain:float = 1.5
    reward_rank:float = .8
    reward_information:float = .07
    ppo_epochs:int = 2
    horizon:int = 20
    dkt_epochs:int = 15
    updates:int = 50
    episodes_update:int = 20
    validation_interval_full:int = 5
    bootstrap_n:int = 2000
    shared_prior:bool = True
    shared_prior_entropy:float = -1.0
    shared_prior_difficulty:float = .5
    dkt_validation_fraction:float = .20
    policy_selection_fraction_full:float = .60
    difficulty_source:str = 'train_only'
    smoke:bool = False
    train_students:int = 0
    val_students:int = 0

C = Config()
C.smoke = MODE in ('smoke', 'kc_smoke')
if C.smoke:
    C.dkt_epochs = 2
    C.updates = 2
    C.episodes_update = 2
    C.ppo_epochs = 1
    C.train_students = 140
    C.val_students = 24
    C.bootstrap_n = 300

DEVICE = torch.device('cpu')  # Interaksi PPO per soal lebih cocok untuk CPU.
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    try:torch.set_num_threads(1)
    except RuntimeError:pass
seed_everything(C.seed)

def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1048576),b''): h.update(block)
    return h.hexdigest()

def save_json(path, obj):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix(path.suffix+'.partial')
    tmp.write_text(json.dumps(obj,indent=2,default=str,ensure_ascii=False),encoding='utf8')
    os.replace(tmp,path)

def save_torch(path,obj):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix(path.suffix+'.partial')
    torch.save(obj,tmp);os.replace(tmp,path)

if IN_COLAB and UPLOAD_DARI_LAPTOP:
    from google.colab import files
    for name,data in files.upload().items():
        if not name.lower().endswith('.zip'):raise ValueError('Hanya ZIP asli DBE-KT22')
        (INPUT/name).write_bytes(data)

print('MODE:',MODE,' | CPU threads:',torch.get_num_threads())
print('Lokasi dataset:',INPUT)
print('Lokasi hasil:',WORKSPACE)
print('DKT:',DKT_DIR)
if MODE in ('report', 'public'): print('MODE LAPORAN: tidak menjalankan training, inferensi ataupun evaluasi test.')
print('Final test unlocked:',FINAL_TEST_UNLOCK)

## 1.3 Konfigurasi yang tidak boleh dituning menggunakan final test

| Parameter | Smoke | Full/multi-seed |
|---|---:|---:|
| DKT epochs | 2 | 15 |
| PPO updates/model | 2 | 50 |
| Episode/update | 2 | 20 |
| PPO epoch/update | 1 | 2 |
| Policy checkpoint selection | 24 mahasiswa | 74 mahasiswa |
| Audit independen dari pemilihan checkpoint | 100 mahasiswa (20 digunakan untuk laporan ringkas) | 50 mahasiswa |

Output: `MyDrive/Riset_MARL_Rerun/<RUN_ID>/`. Pemisahan mode mencegah hasil smoke menimpa checkpoint full.

# 2. DBE-KT22: audit dan pemisahan mahasiswa

## 2.1 Membaca ZIP asli, interaksi kronologis, relasi soal–KC

Semua interaksi diurutkan menurut `student_id`, `start_time`, dan `id`. Duplikasi respons untuk soal yang sama oleh satu mahasiswa dibuang dengan menyimpan respons pertama. Hanya mahasiswa dengan minimal 28 interaksi unik yang masuk eksperimen.

**Penting:** pengujian CAT dilakukan sebagai replay offline. Soal yang pernah dikerjakan mahasiswa menjadi kumpulan aksi, bukan bank soal interaktif yang mempunyai label jawaban potensial untuk semua soal. Ini membatasi interpretasi generalisasi ke CAT dunia nyata.

In [ ]:
# Kode eksperimen asli; dilewati pada MODE='report' tanpa training.
if MODE not in ('report', 'public'):
    zip_candidates = sorted(INPUT.glob('*.zip'))
    if len(zip_candidates) == 0:
        raise FileNotFoundError(
            f'Dataset belum tersedia. Buka Google Drive > MyDrive > Riset_MARL > input, '
            f'unggah 1 ZIP ASLI DBE-KT22 (jangan diekstrak), '
            f'kemudian jalankan Run all kembali. Folder: {INPUT}'
        )
    if len(zip_candidates) > 1:
        raise ValueError(f'Folder input harus berisi TEPAT SATU ZIP dataset asli; ditemukan {len(zip_candidates)}: {zip_candidates}')
    dataset_zip = zip_candidates[0]
    print('Sumber dataset:',dataset_zip)
    print('SHA256:',hashlib.sha256(dataset_zip.read_bytes()).hexdigest())
    with zipfile.ZipFile(dataset_zip) as outer:
        nested = [n for n in outer.namelist() if n.endswith('datafiles_100102_csv.zip')]
        if len(nested)!=1:raise ValueError('Arsip bukan ZIP DBE-KT22 asli (datafiles_100102_csv.zip tidak ditemukan)')
        with zipfile.ZipFile(io.BytesIO(outer.read(nested[0]))) as datazip:
            def getcsv(name): return pd.read_csv(io.BytesIO(datazip.read(name)),low_memory=False)
            tx = getcsv('Transaction.csv')
            relation = getcsv('Question_KC_Relationships.csv')
            question_info = getcsv('Questions.csv')[['id','difficulty']]  # metadata audit saja, tidak dipakai sebagai difficulty
    required = {'student_id','question_id','start_time','answer_state','id'}
    assert required <= set(tx.columns)
    tx['start_time'] = pd.to_datetime(tx['start_time'],utc=True,errors='coerce')
    tx = tx.dropna(subset=['student_id','question_id','start_time','answer_state']).copy()
    tx = tx.sort_values(['student_id','start_time','id'],kind='stable')
    tx = tx.drop_duplicates(['student_id','question_id'],keep='first')
    question_ids = sorted(set(relation.question_id.astype(int)) & set(tx.question_id.astype(int)))
    k_raw = sorted(set(relation.loc[relation.question_id.isin(question_ids),'knowledgecomponent_id'].astype(int)))
    q_to_i = {q:i for i,q in enumerate(question_ids)}
    k_to_i = {k:i for i,k in enumerate(k_raw)}
    Q,K = len(question_ids),len(k_raw)
    qkc = np.zeros((Q,K),dtype=np.float32)
    for row in relation.itertuples(index=False):
        if int(row.question_id) in q_to_i and int(row.knowledgecomponent_id) in k_to_i:
            qkc[q_to_i[int(row.question_id)],k_to_i[int(row.knowledgecomponent_id)]]=1
    qdiff=np.zeros(Q,dtype=np.float32)  # diisi dari data train-only setelah split
    users={}
    for sid,df in tx.groupby('student_id',sort=False):
        seq=[(q_to_i[int(q)],int(y)) for q,y in zip(df.question_id,df.answer_state) if int(q) in q_to_i]
        if len(seq)>=C.test_length+8:users[str(sid)]=seq
    assert users and np.all(qkc.sum(axis=1)>0)
    print(f'Jumlah pengguna layak={len(users)}, soal={Q}, KC={K}, respons unik={sum(map(len,users.values()))}')
    print('Kelas positif:',round(np.mean([y for seq in users.values() for _,y in seq]),4))
    print('Rentang panjang:',min(map(len,users.values())),max(map(len,users.values())))
    # Split berdasarkan mahasiswa disjoint: train / validation / locked test.
    rng = np.random.default_rng(C.split_seed)
    all_sids = np.array(sorted(users)); rng.shuffle(all_sids)
    n_train, n_val = int(0.70*len(all_sids)), int(0.15*len(all_sids))
    SPLIT = {'train': all_sids[:n_train].tolist(),
             'val': all_sids[n_train:n_train+n_val].tolist(),
             'test': all_sids[n_train+n_val:].tolist()}
    assert not(set(SPLIT['train']) & set(SPLIT['val']))
    assert not(set(SPLIT['train']) & set(SPLIT['test']))
    assert not(set(SPLIT['val']) & set(SPLIT['test']))

    # Validation dibagi TANPA test: DKT checkpoint, policy checkpoint, policy audit.
    # Walaupun hasil v2 telah dilihat, protokol v3 tidak memanfaatkan target 'test'.
    n_dkt_val=max(10,int(round(len(SPLIT['val'])*C.dkt_validation_fraction)))
    DKT_VAL_IDS=SPLIT['val'][:n_dkt_val]
    POLICY_VAL_IDS=SPLIT['val'][n_dkt_val:]
    n_policy_select=(C.val_students if C.val_students else
                     max(12,int(round(len(POLICY_VAL_IDS)*C.policy_selection_fraction_full))))
    assert 0<n_policy_select<len(POLICY_VAL_IDS)
    POLICY_SELECT_IDS=POLICY_VAL_IDS[:n_policy_select]
    POLICY_AUDIT_IDS=POLICY_VAL_IDS[n_policy_select:]
    assert not(set(DKT_VAL_IDS) & set(POLICY_SELECT_IDS))
    assert not(set(DKT_VAL_IDS) & set(POLICY_AUDIT_IDS))
    assert not(set(POLICY_SELECT_IDS) & set(POLICY_AUDIT_IDS))

## 2.2 Pembagian mahasiswa dan rancangan replay

Pemisahan menggunakan **student ID**, bukan baris respons. Kelompok validation dibagi menjadi: DKT validation, policy selection, dan policy audit. Untuk setiap mahasiswa, 20% interaksi terakhir menjadi **target diagnosis** yang tidak tersedia untuk aksi agen. Bagian sebelumnya menjadi kumpulan soal yang dapat direplay.

**Difficulty soal** dihitung dari hasil interaksi **mahasiswa training saja** (Beta(2,2) smoothing); metadata `Questions.csv` tetap dibaca untuk audit tetapi tidak masuk fitur model. Ini lebih ketat daripada smoke kandidat terdahulu, sehingga hasil angka sebelumnya tidak dijamin sama.

In [ ]:
# Kode eksperimen asli; dilewati pada MODE='report' tanpa training.
if MODE not in ('report', 'public'):
    # Last 20% chronological logs are target prediction; prefix is offline CAT action pool.
    def student_case(sid):
        seq=users[sid];n_target=max(8,int(round(C.target_frac*len(seq))))
        target,active=seq[-n_target:],seq[:-n_target]
        assert len(active)>=C.test_length and len(target)>=8
        assert set(q for q,_ in active).isdisjoint(set(q for q,_ in target))
        return {'sid':sid,'active':active,'target':target}

    CASES={split:[student_case(sid) for sid in ids] for split,ids in SPLIT.items()}
    CASES['policy_select']=[student_case(s) for s in POLICY_SELECT_IDS]
    CASES['policy_audit']=[student_case(s) for s in POLICY_AUDIT_IDS]
    CASES['dkt_val']=[student_case(s) for s in DKT_VAL_IDS]
    CASE_LOOKUP={c['sid']:c for v in CASES.values() for c in v}

    assert not(set(DKT_VAL_IDS) & set(c['sid'] for c in CASES['policy_select']))
    save_json(TABLES/'split_manifest_private.json', {
        'split_seed':C.split_seed,'student_ids':SPLIT,
        'dkt_validation_ids':DKT_VAL_IDS,'ppo_selection_ids':POLICY_SELECT_IDS,
        'ppo_audit_ids':POLICY_AUDIT_IDS,
        'target_fraction':C.target_frac,'length':C.test_length,
        'note':'validation subgroups are disjoint from each other and held-out test'})
    print('Jumlah train/val/test:',{s:len(SPLIT[s]) for s in SPLIT})
    print('DKT-val / PPO-select / PPO-audit:',len(DKT_VAL_IDS),len(POLICY_SELECT_IDS),len(POLICY_AUDIT_IDS))
    print('Minimal active=',min(len(c['active']) for v in CASES.values() for c in v))

    # Item difficulty train-only, dihitung hanya dari prefix mahasiswa training.
    correct = np.zeros(Q, dtype=np.float64)
    num = np.zeros(Q, dtype=np.float64)
    for sid in SPLIT['train']:
        for question, answer in CASE_LOOKUP[sid]['active']:
            correct[question] += float(answer)
            num[question] += 1.0
    qdiff[:] = ((num - correct) + 2.0) / (num + 4.0)
    assert (qdiff >= 0).all() and (qdiff <= 1).all()
    print('Kesulitan train-only (min / maks):',round(float(qdiff.min()),3),round(float(qdiff.max()),3))
    print('Old held-out test tetap TERKUNCI untuk penelitian ulang ini:',len(CASES['test']),'mahasiswa')

## 2.3 Validasi protokol data

Simpan manifest dataset dan split; pastikan tidak ada mahasiswa yang muncul dalam lebih dari satu kelompok. Semua pipeline pelatihan di bawah dibatasi pada `train` untuk update parameter dan `policy_select` untuk pemilihan checkpoint PPO. `policy_audit` hanya untuk laporan diagnostik.

In [ ]:
# Kode eksperimen asli; dilewati pada MODE='report' tanpa training.
if MODE not in ('report', 'public'):
    DATASET_SHA = sha256(dataset_zip)
    SPLIT_PUBLIC = {k:len(v) for k,v in SPLIT.items()}
    SPLIT_PUBLIC.update({'dkt_val':len(DKT_VAL_IDS),'policy_select':len(POLICY_SELECT_IDS),'policy_audit':len(POLICY_AUDIT_IDS)})
    SPLIT_SHA = hashlib.sha256(json.dumps({'SPLIT':SPLIT,'DKT_VAL_IDS':DKT_VAL_IDS,
        'POLICY_SELECT_IDS':POLICY_SELECT_IDS,'POLICY_AUDIT_IDS':POLICY_AUDIT_IDS},sort_keys=True).encode()).hexdigest()
    DIFFICULTY_SHA=hashlib.sha256(qdiff.tobytes()).hexdigest()
    assert len(set(SPLIT['train'])&set(SPLIT['test']))==0
    assert len(set(POLICY_SELECT_IDS)&set(POLICY_AUDIT_IDS))==0
    assert all(len(c['active'])>=C.test_length for group in CASES.values() for c in group)
    assert all(set(q for q,_ in c['active']).isdisjoint(q for q,_ in c['target']) for group in CASES.values() for c in group)
    meta={'dataset_sha256':DATASET_SHA,'split_sha256':SPLIT_SHA,'difficulty_sha256':DIFFICULTY_SHA,
          'students':SPLIT_PUBLIC,'questions':Q,'knowledge_components':K,
          'config':asdict(C),'difficulty_source':C.difficulty_source,'mode':MODE,
          'offline_replay':True,'known_prior_test_exposure':True}
    save_json(REPORTS/'experiment_identity.json',meta)
    display(pd.DataFrame([{'subset':k,'students':v} for k,v in SPLIT_PUBLIC.items()]))
    print('Data SHA',DATASET_SHA[:16],'Split SHA',SPLIT_SHA[:16], 'Difficulty SHA', DIFFICULTY_SHA[:16])

# 3. Deep Knowledge Tracing (DKT)

## 3.1 Definisi model dan prosedur pelatihan

DKT GRU memproses `(soal, respons)` secara temporal dan memperkirakan peluang jawaban benar pada soal berikutnya. Target yang digunakan untuk memberi reward kebijakan di data training **tidak** diberikan kepada DKT dalam training encoder. State DKT diperbarui hanya setelah respons replay dari soal terpilih tersedia.

- `MODE='smoke'`: DKT dilatih **dari awal** selama dua epoch; checkpoint disimpan di `dkt/smoke/`.
- `MODE='full'`: DKT dilatih **dari awal** selama 15 epoch; checkpoint disimpan di `dkt/full/`.
- `multiseed`, `ablation`, `freeze`, `final`: hanya memakai checkpoint `dkt/full/`; tidak melatih ulang DKT.

In [ ]:
# Kode eksperimen asli; dilewati pada MODE='report' tanpa training.
if MODE not in ('report', 'public'):
    class SharedDKT(nn.Module):
        def __init__(self, qn,kn,qkc,hidden=48,emb=32):
            super().__init__()
            self.qn,self.hidden = qn,hidden
            self.token_embedding = nn.Embedding(2*qn+1,emb,padding_idx=0)
            self.gru = nn.GRU(emb,hidden,batch_first=True)
            self.question_embedding = nn.Embedding(qn,hidden)
            self.kc_proj = nn.Linear(kn,hidden,bias=False)
            self.head = nn.Sequential(nn.Linear(3*hidden,hidden),nn.ReLU(),nn.Linear(hidden,1))
            self.register_buffer('qkc',torch.as_tensor(qkc,dtype=torch.float32))
        def token(self,q,y): return 1 + q.long() + self.qn*y.long()
        def predict_logits(self,h,q):
            # q shape [N] or [B,T]; h shape [N,H] or [B,T,H].
            item = self.question_embedding(q) + self.kc_proj(self.qkc[q])
            return self.head(torch.cat([h,item,h*item],dim=-1)).squeeze(-1)
        def forward(self,tokens,questions):
            # tokens[:,0] adalah padding penanda kondisi sebelum soal pertama.
            # h_t setelah t-1 respons; identik dengan zero_state() dan step() saat CAT.
            if tokens.shape[1] > 1:
                h_after,_ = self.gru(self.token_embedding(tokens[:,1:]))
                h = torch.cat([torch.zeros_like(h_after[:,:1]),h_after],dim=1)
            else:
                h = torch.zeros(tokens.shape[0],1,self.hidden,device=tokens.device)
            return self.predict_logits(h,questions)
        @torch.no_grad()
        def step(self,h,q,y):
            qt=torch.tensor([q],dtype=torch.long,device=h.device)
            yt=torch.tensor([y],dtype=torch.long,device=h.device)
            _,next_h=self.gru(self.token_embedding(self.token(qt,yt)).unsqueeze(1),h.unsqueeze(0))
            return next_h.squeeze(0)
        @torch.no_grad()
        def probs(self,h,qs):
            q = torch.as_tensor(qs,dtype=torch.long,device=h.device)
            hh = h.expand(len(q),-1)
            return torch.sigmoid(self.predict_logits(hh,q)).detach().cpu().numpy()
        def zero_state(self): return torch.zeros(1,self.hidden,device=self.qkc.device)

    DKT = SharedDKT(Q,K,qkc,C.hidden,C.emb).to(DEVICE)

    def batches_for_dkt(sids,batch_size=32,shuffle=False,seed=42):
        rr=np.random.default_rng(seed)
        ss=list(sids)
        if shuffle: rr.shuffle(ss)
        for lo in range(0,len(ss),batch_size):
            sub=ss[lo:lo+batch_size]
            L=max(len(users[sid]) for sid in sub)
            tokens=np.zeros((len(sub),L),dtype=np.int64)
            qs=np.zeros((len(sub),L),dtype=np.int64)
            yy=np.zeros((len(sub),L),dtype=np.float32)
            mask=np.zeros((len(sub),L),dtype=np.float32)
            for i,sid in enumerate(sub):
                series=CASE_LOOKUP[sid]['active']  # target withheld from ALL DKT training/validation passes
                qq=np.array([q for q,_ in series]); lab=np.array([y for _,y in series])
                l=len(series)
                qs[i,:l]=qq; yy[i,:l]=lab;mask[i,:l]=1
                if l>1: tokens[i,1:l]=1+qq[:-1]+Q*lab[:-1]
            yield (torch.as_tensor(tokens,device=DEVICE),torch.as_tensor(qs,device=DEVICE),
                   torch.as_tensor(yy,device=DEVICE),torch.as_tensor(mask,device=DEVICE))

    def dkt_dataset_bce(sids):
        DKT.eval(); total, num=0.0,0
        with torch.no_grad():
            for t,q,y,m in batches_for_dkt(sids):
                loss=F.binary_cross_entropy_with_logits(DKT(t,q),y,reduction='none')
                total+=float((loss*m).sum());num+=int(m.sum())
        return total/max(num,1)

    # Pakai validation saja untuk pemilihan checkpoint DKT.

## 3.2 Training DKT, pemilihan checkpoint, dan resume

Checkpoint terbaik dipilih dari **BCE DKT validation**, tidak dari final test. Setelah runtime terputus, checkpoint progres memuat model dan optimizer agar training dapat diteruskan. Seed dikendalikan per epoch.

In [ ]:
# Kode eksperimen asli; dilewati pada MODE='report' tanpa training.
if MODE not in ('report', 'public'):
    DKT_BEST = DKT_DIR/'dkt_best.pt'
    DKT_PROGRESS = DKT_DIR/'dkt_progress.pt'
    DKT_HISTORY_PATH = DKT_DIR/'dkt_history.csv'
    DKT_SIGNATURE=hashlib.sha256(json.dumps({'data':DATASET_SHA,'split':SPLIT_SHA,
        'qdiff':DIFFICULTY_SHA,'hidden':C.hidden,'emb':C.emb,'dkt_lr':C.dkt_lr,
        'epochs':C.dkt_epochs,'smoke':C.smoke},sort_keys=True).encode()).hexdigest()

    def train_or_load_dkt():
        train_mode=MODE in ('smoke','full','kc_smoke','kc_audit')
        if not train_mode:
            if not DKT_BEST.exists():
                raise FileNotFoundError(f'DKT full belum tersedia: {DKT_BEST}. Jalankan MODE="full" dahulu.')
            identity=DKT_DIR/'dkt_identity.json'
            if not identity.exists() or json.loads(identity.read_text())['signature'] != DKT_SIGNATURE:
                raise RuntimeError('DKT identity berbeda; hentikan. Periksa dataset/split/config. Jangan hapus checkpoint.')
            DKT.load_state_dict(torch.load(DKT_BEST,map_location=DEVICE,weights_only=True))
            DKT.eval()
            return pd.read_csv(DKT_HISTORY_PATH)
        seed_everything(C.seed)
        opt=torch.optim.AdamW(DKT.parameters(),lr=C.dkt_lr,weight_decay=1e-4)
        best=float('inf');start=1;log=[]
        if DKT_PROGRESS.exists():
            data=torch.load(DKT_PROGRESS,map_location=DEVICE,weights_only=True)
            if data['signature']!=DKT_SIGNATURE: raise RuntimeError('Identitas DKT berubah. Jangan timpa checkpoint.')
            DKT.load_state_dict(data['model'])
            opt.load_state_dict(data['optimizer'])
            best=float(data['best']);start=int(data['epoch'])+1;log=data['history']
            print('Resume DKT mulai epoch',start)
        train_ids=SPLIT['train'][:C.train_students] if C.train_students else SPLIT['train']
        for ep in range(start,C.dkt_epochs+1):
            seed_everything(C.seed+ep*7919)
            DKT.train();cum=0;count=0
            for t,q,y,m in batches_for_dkt(train_ids,shuffle=True,seed=C.seed+ep):
                loss=(F.binary_cross_entropy_with_logits(DKT(t,q),y,reduction='none')*m).sum()/m.sum().clamp_min(1)
                opt.zero_grad();loss.backward();nn.utils.clip_grad_norm_(DKT.parameters(),1.0)
                opt.step();cum+=float(loss.detach());count+=1
            val=dkt_dataset_bce(DKT_VAL_IDS)
            log.append({'epoch':ep,'train_batch_bce':cum/max(1,count),'val_bce':val})
            if val<best:
                best=val;save_torch(DKT_BEST,DKT.state_dict())
            save_torch(DKT_PROGRESS,{'signature':DKT_SIGNATURE,'model':DKT.state_dict(),
                'optimizer':opt.state_dict(),'history':log,'best':best,'epoch':ep})
            pd.DataFrame(log).to_csv(DKT_HISTORY_PATH,index=False)
            print(f'DKT epoch {ep}/{C.dkt_epochs} | train={cum/max(1,count):.4f} | val={val:.4f}')
        if not DKT_BEST.exists():raise RuntimeError('Checkpoint DKT terbaik tidak ditemukan')
        DKT.load_state_dict(torch.load(DKT_BEST,map_location=DEVICE,weights_only=True))
        DKT.eval()
        save_json(DKT_DIR/'dkt_identity.json',{'signature':DKT_SIGNATURE,'checkpoint_sha256':sha256(DKT_BEST),
           'data_sha256':DATASET_SHA,'split_sha256':SPLIT_SHA,'config':asdict(C)})
        return pd.DataFrame(log)

    DKT_HISTORY=train_or_load_dkt()
    print('DKT checkpoint SHA:',sha256(DKT_BEST))
    display(DKT_HISTORY.tail())

## 3.3 Diagnostik DKT dan konsistensi GRU

Periksa bahwa inference bertahap setelah sejumlah respons menghasilkan state dan prediksi yang sama dengan forward GRU pada urutan tersebut. Simpan kurva train/validation BCE sebagai PNG, SVG, dan CSV.

In [ ]:
# Kode eksperimen asli; dilewati pada MODE='report' tanpa training.
if MODE not in ('report', 'public'):
    # Urutan uji sama dengan respons kronologis yang telah dipisah.
    test_seq=CASES['dkt_val'][0]['active'][:8]
    h=DKT.zero_state()
    with torch.no_grad():
        for q,y in test_seq: h=DKT.step(h,q,y)
        q_eval=test_seq[-1][0]
        p_step=float(DKT.probs(h,[q_eval])[0])
        # Manual GRU token sama dengan step berulang, bukan final-test target.
        toks=torch.tensor([1+q+Q*y for q,y in test_seq],dtype=torch.long,device=DEVICE).unsqueeze(0)
        z,_=DKT.gru(DKT.token_embedding(toks))
        p_seq=float(torch.sigmoid(DKT.predict_logits(z[:,-1,:],torch.tensor([q_eval],device=DEVICE)))[0])
    assert abs(p_step-p_seq)<1e-5,(p_step,p_seq)
    fig,ax=plt.subplots(figsize=(7.8,3.5))
    ax.plot(DKT_HISTORY.epoch,DKT_HISTORY.train_batch_bce,marker='o',label='Training BCE')
    ax.plot(DKT_HISTORY.epoch,DKT_HISTORY.val_bce,marker='s',label='Validation BCE')
    ax.set(xlabel='Epoch',ylabel='BCE',title='Kurva training DKT')
    ax.legend();ax.grid(alpha=.2);fig.tight_layout()
    fig.savefig(FIGURES/'01_dkt_learning_curve.png',dpi=170)
    fig.savefig(FIGURES/'01_dkt_learning_curve.svg')
    display(fig);plt.close(fig)

# 4. CAT offline: definisi state, action, reward dan metrik

## 4.1 State dan action

- State: `h_DKT`, KC yang telah dicakup, dan fraksi panjang tes.
- Global state untuk critic: state + ketersediaan KC yang tersisa.
- Fitur soal: peluang benar DKT, entropi, kebaruan KC, dan difficulty train-only.
- Action: indeks soal dari interaksi yang pernah teramati, tanpa pengulangan.
- MARL: Planning memilih strategi, KC Agent memilih konsep layak, Item Agent memilih soal valid yang memuat konsep tersebut.

## 4.2 Reward multi-objective berbasis data training

\[r_t=1.0\,\Delta KC+1.5\,\Delta BCE+0.8\,\Delta Rank+0.07\,H(q_t)/20\]

BCE gain dan ranking gain dihitung terhadap interaksi target **mahasiswa training saja**. Untuk policy selection/audit/final, target digunakan **hanya saat menghitung metrik**, bukan reward atau pemilihan soal. `rank` adalah surrogate sigmoid untuk ranking prediksi benar versus salah; bukan AUC aktual yang dipakai sebagai reward.

## 4.3 Metrik

**KC Available** = KC yang tercakup / KC yang tersedia dalam kumpulan soal mahasiswa. **KC Global** = KC yang tercakup / seluruh KC dataset. **AUC** dihitung dengan menggabungkan prediksi jawaban target semua mahasiswa dalam kelompok evaluasi; **BCE** menghitung rata-rata loss semua target. `length` diharapkan 20 untuk seluruh metode.

In [ ]:
# Kode eksperimen asli; dilewati pada MODE='report' tanpa training.
if MODE not in ('report', 'public'):
    class OfflineCAT:
        def __init__(self,case):
            self.sid=case['sid']
            self.active=dict(case['active'])
            self.target_q=np.array([q for q,_ in case['target']],dtype=np.int64)
            self.target_y=np.array([y for _,y in case['target']],dtype=np.float32)
            self.eligible=np.zeros(Q,dtype=bool)
            self.eligible[list(self.active)]=True
            self.available_kc = np.asarray(qkc[self.eligible].sum(axis=0)>0,dtype=bool)
            self.total_eligible=int(self.eligible.sum())
            self.reset()
        def reset(self):
            self.h=DKT.zero_state().detach()
            self.remaining=self.eligible.copy()
            self.covered=np.zeros(K,dtype=bool)
            self.n=0
            self.chosen=[]
            return self.obs()
        def obs(self):
            hh=self.h.detach().cpu().numpy().reshape(-1)
            frac=np.array([self.n/C.test_length],dtype=np.float32)
            state=np.concatenate([hh,self.covered.astype(np.float32),frac]).astype(np.float32)
            remk=(qkc[self.remaining].sum(axis=0)/max(self.total_eligible,1)).astype(np.float32)
            global_obs=np.concatenate([state,remk]).astype(np.float32)
            # Semata-mata dari h dan metadata item, bukan dari jawaban target.
            p=DKT.probs(self.h,np.arange(Q)).astype(np.float32)
            entropy=-(p*np.log(p+1e-7)+(1-p)*np.log(1-p+1e-7))/np.log(2)
            novelty=(qkc*(~self.covered)[None,:]).sum(axis=1)/np.maximum(1,qkc.sum(axis=1))
            features=np.column_stack([p,entropy,novelty,qdiff]).astype(np.float32)
            feasible_kc=np.asarray(qkc[self.remaining].sum(axis=0)>0,dtype=bool)
            # Project the fixed KC scores onto each item's linked, feasible KCs.
            # Fixed Balanced weights mirror the hierarchical prior's Balanced option.
            kc_counts = qkc[self.remaining].sum(axis=0)
            kc_uncertainty = (qkc.T @ (entropy * self.remaining)) / np.maximum(kc_counts, 1)
            kc_bias = .44 * (~self.covered) + .40 * kc_uncertainty
            item_kc_bias = (qkc @ kc_bias) / qkc.sum(axis=1)
            matched_prior = (item_kc_bias + .65 * novelty - .07 * entropy).astype(np.float32)
            return {'matched_prior':matched_prior, 'state':state,'global_obs':global_obs,'features':features,'mask_q':self.remaining.copy(),
                    'mask_k':feasible_kc,'covered':self.covered.copy()}
        def target_bce(self):
            pred=np.clip(DKT.probs(self.h,self.target_q),1e-6,1-1e-6)
            return float(np.mean(-self.target_y*np.log(pred)-(1-self.target_y)*np.log(1-pred)))
        def act(self,q,training=False):
            q=int(q)
            if q<0 or q>=Q or not self.remaining[q]:raise ValueError(f'Pilihan tidak tersedia: {q}')
            before=self.target_bce() if training else 0.0
            pq=float(DKT.probs(self.h,[q])[0])
            inf=float(-(pq*np.log(pq+1e-7)+(1-pq)*np.log(1-pq+1e-7))/np.log(2))
            new=np.asarray((qkc[q]>0)&(~self.covered),dtype=bool)
            coverage_gain=float(new.sum()) / max(1,int(self.available_kc.sum()))
            y=int(self.active[q]);self.h=DKT.step(self.h,q,y)
            self.covered |= (qkc[q]>0)
            self.remaining[q]=False;self.chosen.append(q);self.n+=1
            done=self.n>=C.test_length
            if training:
                bce_gain=np.clip(before-self.target_bce(),-0.3,0.3)
                reward=C.reward_coverage*coverage_gain + C.reward_bce_gain*bce_gain + C.reward_information*inf/C.test_length
            else:
                reward=0.0
            return self.obs(),float(reward),done
        def result(self):
            pred=np.clip(DKT.probs(self.h,self.target_q),1e-6,1-1e-6)
            auc=roc_auc_score(self.target_y,pred) if len(set(self.target_y.tolist()))==2 else np.nan
            return {'sid':self.sid,'kc_global':float(self.covered.mean()),
                    'kc_available':float(self.covered.sum()/max(1,self.available_kc.sum())),
                    'bce':float(np.mean(-self.target_y*np.log(pred)-(1-self.target_y)*np.log(1-pred))),
                    'auc_student':float(auc),'length':self.n,'y':self.target_y.copy(),'p':pred.copy()}

    example=OfflineCAT(CASES['train'][0]);o=example.reset()
    assert o['mask_q'].sum()>=C.test_length
    assert not np.any(np.isin(example.target_q,np.where(o['mask_q'])[0]))
    print('State:',o['state'].shape,'Global:',o['global_obs'].shape,'Features:',o['features'].shape)
    OldOfflineCAT=OfflineCAT
    from scipy.special import expit
    class OfflineCAT(OldOfflineCAT):
     def rank_surrogate(self):
      p=DKT.probs(self.h,self.target_q)
      pos=p[self.target_y==1];neg=p[self.target_y==0]
      if len(pos)==0 or len(neg)==0:return 0.0
      return float(expit((pos[:,None]-neg[None,:])/0.15).mean())
     def act(self,q,training=False):
      if not training:return super().act(q,False)
      prev=self.target_bce(); rank_before=self.rank_surrogate()
      oldcov=C.reward_coverage;oldbce=C.reward_bce_gain;oldinf=C.reward_information
      # base routine supplies consistent action+state transition and coverage/entropy gain
      obs,rew,done=super().act(q,True)
      rank_gain=float(np.clip(self.rank_surrogate()-rank_before,-0.20,0.20))
      return obs,float(rew+C.reward_rank*rank_gain),done

## 4.4 Pemeriksaan lingkungan

Uji ketersediaan aksi, larangan target dalam kandidat soal, reward finite, dan panjang tes. Uji ini **tidak** memakai mahasiswa final test.

In [ ]:
# Kode eksperimen asli; dilewati pada MODE='report' tanpa training.
if MODE not in ('report', 'public'):
    probe=OfflineCAT(CASES['train'][0]);obs=probe.reset()
    assert obs['state'].shape==(C.hidden+K+1,)
    assert obs['features'].shape==(Q,4)
    assert obs['mask_q'].sum()>=C.test_length
    assert not(set(probe.target_q)&set(np.flatnonzero(obs['mask_q'])))
    first=int(np.flatnonzero(obs['mask_q'])[0]);_,reward,_=probe.act(first,training=True)
    assert np.isfinite(reward) and probe.n==1 and not probe.remaining[first]
    print('CAT unit test berhasil; reward training pertama:',round(reward,5))

# 5. Dua baseline dan tiga agen MARL

## 5.1 Definisi arsitektur

**RL-Flat:** logits langsung untuk seluruh soal dari state DKT. **RL-Feat:** logits untuk seluruh soal dari state dan fitur soal. **MARL:** Planning Agent → KC Selection Agent → Item Selection Agent. Ketiga aktor MARL dioptimasi bersama dengan cooperative PPO dan satu critic bersama. Ketiga kebijakan menggunakan **shared action prior** yang sama; ablation dapat menonaktifkannya secara eksplisit.

Desain kebijakan diagnostik digunakan sebagai dasar, tetapi **tidak ada nama varian eksperimen dalam objek model atau nama output**. Nama model utama cukup `marl`.

In [ ]:
# Kode eksperimen asli; dilewati pada MODE='report' tanpa training.
if MODE not in ('report', 'public'):
    def tofloat(x):return torch.as_tensor(x,dtype=torch.float32,device=DEVICE)
    def mask_dist(logits,mask):
        m=torch.as_tensor(mask,dtype=torch.bool,device=DEVICE)
        if not torch.any(m):raise ValueError('Semua aksi terlarang')
        return torch.distributions.Categorical(logits=logits.masked_fill(~m,-1e9))

    class FlatActor(nn.Module):
        def __init__(self,dim):
            super().__init__()
            self.net=nn.Sequential(nn.Linear(dim,96),nn.Tanh(),nn.Linear(96,Q))
        def forward(self,state,feat):return self.net(state)

    class FeatActor(nn.Module):
        def __init__(self,dim):
            super().__init__()
            self.state=nn.Sequential(nn.Linear(dim,64),nn.Tanh())
            self.feature=nn.Linear(4,64)
            self.item=nn.Embedding(Q,64)
            self.score=nn.Sequential(nn.Tanh(),nn.Linear(64,1))
        def forward(self,state,features):
            x=self.state(state).unsqueeze(0)+self.feature(features)+self.item.weight
            return self.score(x).squeeze(-1)


    class PlanKCItemActor(nn.Module):
     def __init__(self,dim):
      super().__init__();
      self.plan=nn.Sequential(nn.Linear(dim+4,80),nn.Tanh(),nn.Linear(80,3))
      self.kc=nn.Sequential(nn.Linear(dim+3+4,96),nn.Tanh(),nn.Linear(96,K))
      self.state_proj=nn.Linear(dim,64)
      self.feature_proj=nn.Linear(7,64)
      self.item_embeddings=nn.Embedding(Q,64)
      self.plan_embeddings=nn.Embedding(3,64)
      self.kc_embeddings=nn.Embedding(K,64)
      self.score=nn.Sequential(nn.Tanh(),nn.Linear(64,1))
     def summary(self,obs):
      f=tofloat(obs['features']);mask=tofloat(obs['mask_q'].astype(np.float32))
      p=f[:,0]; ent=f[:,1]
      return torch.stack([f[:,2].mul(mask).sum()/mask.sum().clamp_min(1),
       ent.mul(mask).sum()/mask.sum().clamp_min(1),
       tofloat(obs['covered'].astype(float)).mean(),
       mask.mean()])
     def plan_logits(self,obs):
      x=torch.cat([tofloat(obs['state']),self.summary(obs)])
      return self.plan(x)
     def kc_logits(self,obs,plan):
      s=tofloat(obs['state']);f=tofloat(obs['features']);mask=tofloat(obs['mask_q'].astype(np.float32))
      ohp=F.one_hot(torch.tensor(plan,device=DEVICE),3).float()
      x=torch.cat([s,ohp,self.summary(obs)])
      logits=self.kc(x)
      kc_count=(tofloat(qkc).T@mask).clamp_min(1)
      uncertainty=(tofloat(qkc).T @ (f[:,1]*mask))/kc_count
      ncovered=1-tofloat(obs['covered'].astype(np.float32))
      strength=1.0
      return logits if getattr(self,'no_kc_prior',False) else logits+strength*(0.85*ncovered if plan==0 else (0.85*uncertainty if plan==1 else 0.44*ncovered+0.40*uncertainty))
     def item_logits(self,obs,plan,kc):
      st=tofloat(obs['state']);feats=tofloat(obs['features']);
      match=tofloat(qkc[:,kc]);n_new=feats[:,2]
      # 7 per-item diagnostics = probability, entropy, novelty, difficulty,
      # KC match, uncertainty*match, novelty*match.
      f=torch.cat([feats,match[:,None],(feats[:,1]*match)[:,None],(n_new*match)[:,None]],dim=1)
      x=(self.state_proj(st)[None,:] + self.feature_proj(f) + self.item_embeddings.weight
          + self.plan_embeddings(torch.tensor(plan,device=DEVICE))[None,:]
          + self.kc_embeddings(torch.tensor(kc,device=DEVICE))[None,:])
      logits=self.score(x).squeeze(-1)
      # fixed semantic priors, never depend on target answer
      if not getattr(self,'no_semantic_prior',False):
       logits=logits + 0.65*f[:,6] + 0.18*f[:,5] - 0.25*feats[:,1]
      return logits

## 5.2 Joint policy dan critic

Log-probabilitas MARL merupakan jumlah log-prob dari ketiga pemilihan: `log π(plan|state) + log π(kc|state,plan) + log π(item|state,plan,kc)`. Aksi tidak valid di-*mask* sebelum sampling. `no_planning` menggunakan strategi Balanced tetap, bukan menghapus KC/Item Agent.

In [ ]:
# Kode eksperimen asli; dilewati pada MODE='report' tanpa training.
if MODE not in ('report', 'public'):
    class Policy(nn.Module):
        def __init__(self,kind,ablation='none'):
            super().__init__()
            assert kind in ('rl_flat','rl_feat','marl','rl_kc')
            self.kind=kind;self.ablation=ablation
            self.use_action_prior=bool(C.shared_prior and ablation!='no_prior')
            dim=C.hidden+K+1
            if kind=='rl_flat':self.actor=FlatActor(dim)
            elif kind in ('rl_feat','rl_kc'):self.actor=FeatActor(dim)
            else:
                self.actor=PlanKCItemActor(dim)
                self.actor.no_semantic_prior=ablation in ('no_semantic_prior','no_special_priors')
                self.actor.no_kc_prior=ablation=='no_special_priors'
            self.critic=nn.Sequential(nn.Linear(dim+K,96),nn.Tanh(),nn.Linear(96,1))
        def value(self,obs):return self.critic(tofloat(obs['global_obs'])).squeeze(-1)
        def choice(self,obs,greedy=False,actions=None):
            def pick(dist,forced=None):
                a=(int(dist.probs.argmax()) if greedy else int(dist.sample())) if forced is None else int(forced)
                return a,dist.log_prob(torch.tensor(a,device=DEVICE)),dist.entropy()
            if self.kind!='marl':
                logits=self.actor(tofloat(obs['state']),tofloat(obs['features']))
                if self.use_action_prior:
                    logits=logits+C.shared_prior_entropy*tofloat(obs['features'])[:,1]+C.shared_prior_difficulty*tofloat(obs['features'])[:,3]
                if self.kind=='rl_kc':
                    logits=logits+tofloat(obs['matched_prior'])
                dist=mask_dist(logits,obs['mask_q'])
                q,lp,ent=pick(dist,None if actions is None else actions['q'])
                return {'q':q},lp,ent
            if self.ablation=='no_planning':
                plan,l0,e0=2,torch.tensor(0.,device=DEVICE),torch.tensor(0.,device=DEVICE)
            else:
                d0=torch.distributions.Categorical(logits=self.actor.plan_logits(obs))
                plan,l0,e0=pick(d0,None if actions is None else actions['plan'])
            kd=mask_dist(self.actor.kc_logits(obs,plan),obs['mask_k'])
            kc,l1,e1=pick(kd,None if actions is None else actions['kc'])
            qlogits=self.actor.item_logits(obs,plan,kc)
            if self.use_action_prior:
                qlogits=qlogits+C.shared_prior_entropy*tofloat(obs['features'])[:,1]+C.shared_prior_difficulty*tofloat(obs['features'])[:,3]
            qm=obs['mask_q']&(qkc[:,kc]>0)
            qd=mask_dist(qlogits,qm)
            q,l2,e2=pick(qd,None if actions is None else actions['q'])
            return {'plan':plan,'kc':kc,'q':q},l0+l1+l2,e0+e1+e2

## 5.3 Unit test masking, shared DKT, dan gradien

Cek bahwa ketiga agen menghasilkan aksi valid, DKT tetap `frozen` saat training PPO, dan jalur gradien model berbeda sesuai baseline. **Catatan adil:** RL-Flat tidak mempunyai fitur item conditioning, sementara RL-Feat mempunyai fitur. Perbedaan kapasitas parameter dilaporkan secara terbuka.

In [ ]:
# Kode eksperimen asli; dilewati pada MODE='report' tanpa training.
if MODE not in ('report', 'public'):
    obs=OfflineCAT(CASES['train'][1]).reset()
    for name in ('rl_flat','rl_feat','marl','rl_kc'):
        p=Policy(name).to(DEVICE)
        act,lp,ent=p.choice(obs)
        assert obs['mask_q'][act['q']] and torch.isfinite(lp) and torch.isfinite(ent)
        if name=='marl':assert qkc[act['q'],act['kc']]>0
        print(name,'parameters',sum(t.numel() for t in p.parameters()),'example action',act)
    assert not any(x.requires_grad is False for x in Policy('marl').parameters())
    for param in DKT.parameters():param.requires_grad_(False)
    DKT.eval()

# 6. Training PPO, validasi, checkpoint, dan resume

## 6.1 Rollout dan evaluasi

Episode memakai respons teramati (offline replay), tetapi posisi soal **dipilih** oleh kebijakan. Reward diagnosis hanya dihitung pada mahasiswa training. Metrik policy selection dipakai untuk memilih checkpoint; policy audit dipakai untuk membaca generalisasi terhadap mahasiswa validation lain. Semua seed mengikuti anggaran training yang sama.

In [ ]:
# Kode eksperimen asli; dilewati pada MODE='report' tanpa training.
if MODE not in ('report', 'public'):
    def get_cases(split):
        cases=CASES['policy_audit'] if split=='audit' else CASES[split]
        if split=='test' and MODE!='final': raise RuntimeError('Final test terkunci. Akses hanya di MODE=final.')
        if split=='train' and C.train_students: return cases[:C.train_students]
        if split=='val':return CASES['policy_select']
        if split=='audit':return CASES['policy_audit']
        return cases

    @torch.no_grad()
    def evaluate(policy,cases,greedy=True):
        policy.eval();rows=[]
        for case in cases:
            env=OfflineCAT(case);obs=env.reset()
            for t in range(C.test_length):
                acts,_,_=policy.choice(obs,greedy=greedy)
                obs,_,done=env.act(acts['q'],training=False)
                if done:break
            if MODE in ('kc_smoke','kc_audit'):
                assert env.n == C.test_length and len(set(env.chosen)) == C.test_length
                assert all(env.eligible[q] for q in env.chosen)
            rows.append(env.result())
        return rows

    def summarize(rows):
        y=np.concatenate([r['y'] for r in rows]);p=np.concatenate([r['p'] for r in rows])
        return {'kc_available':float(np.mean([r['kc_available'] for r in rows])),
                'kc_global':float(np.mean([r['kc_global'] for r in rows])),
                'auc':float(roc_auc_score(y,p)) if len(np.unique(y))==2 else float('nan'),
                'bce':float(np.mean(-y*np.log(p)-(1-y)*np.log(1-p))),
                'length':float(np.mean([r['length'] for r in rows]))}

    def val_utility(metrics):
        # Tetapkan sebelum pelatihan; hanya val, tanpa akses test.
        # Skala utilitas ini dipakai untuk checkpoint selection, bukan untuk mendefinisikan kemenangan.
        return metrics['kc_available'] + 0.5*metrics['auc'] - 0.5*metrics['bce']

    def run_episode(policy,case):
        env=OfflineCAT(case);obs=env.reset();transition=[]
        for t in range(C.test_length):
            with torch.no_grad():
                actions,logp,_=policy.choice(obs)
                v=float(policy.value(obs))
            next_obs,reward,done=env.act(actions['q'],training=True)
            transition.append({'obs':obs,'actions':actions,'logp_old':float(logp),'old_v':v,
                               'reward':reward,'done':done})
            obs=next_obs
            if done:break
        # Monte-Carlo episode sudah selesai; advantage GAE.
        gae=0.0;next_value=0.0
        for r in reversed(transition):
            delta=r['reward']+C.gamma*next_value*(not r['done'])-r['old_v']
            gae=delta+C.gamma*C.gae_lambda*(not r['done'])*gae
            r['adv']=gae;r['return']=gae+r['old_v'];next_value=r['old_v']
        return transition

    def ppo_update(policy,optimizer,data):
        # Intentionally simple per-trajectory loops for interpretability; can be batched later.
        advantages=np.asarray([r['adv'] for r in data],dtype=np.float32)
        advantages=(advantages-advantages.mean())/(advantages.std()+1e-6)
        ids=np.arange(len(data)); losses=[]
        for ep in range(C.ppo_epochs):
            np.random.shuffle(ids)
            for chunk in np.array_split(ids,max(1,int(np.ceil(len(ids)/48)))):
                pl=[];vl=[];en=[]
                for ix in chunk:
                    r=data[ix]
                    _,new_lp,entropy=policy.choice(r['obs'],actions=r['actions'])
                    ratio=torch.exp(new_lp-torch.tensor(r['logp_old'],device=DEVICE))
                    adv=tofloat(advantages[ix]);s1=ratio*adv;s2=torch.clamp(ratio,1-C.clip,1+C.clip)*adv
                    pl.append(-torch.minimum(s1,s2))
                    vl.append(F.mse_loss(policy.value(r['obs']),tofloat(r['return'])))
                    en.append(entropy)
                loss=torch.stack(pl).mean()+0.5*torch.stack(vl).mean()-0.001*torch.stack(en).mean()
                optimizer.zero_grad();loss.backward();nn.utils.clip_grad_norm_(policy.parameters(),0.5)
                optimizer.step();losses.append(float(loss.detach()))
        return float(np.mean(losses))

## 6.2 PPO vektor dan kesetaraan log-probabilitas

PPO penuh menggunakan mini-batch berbasis vektor agar lebih cepat daripada loop per-item. Rumus clipped PPO sama dengan implementasi smoke. Uji di bawah memeriksa log-prob hasil batch terhadap log-prob `choice(..., actions=...)` dan menghindari perbedaan aturan mask.

In [ ]:
# Kode eksperimen asli; dilewati pada MODE='report' tanpa training.
if MODE not in ('report', 'public'):
    # Batching PPO untuk training penuh: matematika clipped PPO sama,
    # tetapi forward policy untuk seluruh mini-batch menggantikan loop satu aksi.
    def vectorized_ppo_update(policy, optimizer, data):
        advantage=np.asarray([r['adv'] for r in data],dtype=np.float32)
        advantage=(advantage-advantage.mean())/(advantage.std()+1e-6)
        ids=np.arange(len(data)); losses=[]
        qkc_t=tofloat(qkc)
        for epoch in range(C.ppo_epochs):
            np.random.shuffle(ids)
            for ind in np.array_split(ids,max(1,int(np.ceil(len(ids)/48)))):
                rr=[data[int(j)] for j in ind]
                st=tofloat(np.stack([r['obs']['state'] for r in rr]))
                gl=tofloat(np.stack([r['obs']['global_obs'] for r in rr]))
                ft=tofloat(np.stack([r['obs']['features'] for r in rr]))
                qm=torch.as_tensor(np.stack([r['obs']['mask_q'] for r in rr]),device=DEVICE,dtype=torch.bool)
                km=torch.as_tensor(np.stack([r['obs']['mask_k'] for r in rr]),device=DEVICE,dtype=torch.bool)
                av=tofloat(advantage[ind]);oldlp=tofloat([r['logp_old'] for r in rr]); ret=tofloat([r['return'] for r in rr])
                selected_q=torch.tensor([r['actions']['q'] for r in rr],device=DEVICE,dtype=torch.long)
                def dist_mask(logits,mask):
                    return torch.distributions.Categorical(logits=logits.masked_fill(~mask,-1e9))
                if policy.kind=='rl_flat':
                    lg=policy.actor.net(st)
                    if policy.use_action_prior:
                        lg=lg+C.shared_prior_entropy*ft[:,:,1]+C.shared_prior_difficulty*ft[:,:,3]
                    d=dist_mask(lg,qm);lp=d.log_prob(selected_q);entropy=d.entropy()
                elif policy.kind in ('rl_feat','rl_kc'):
                    x=(policy.actor.state(st)[:,None,:]
                       + policy.actor.feature(ft)+policy.actor.item.weight[None,:,:])
                    lg=policy.actor.score(x).squeeze(-1)
                    if policy.kind=='rl_kc':
                        lg=lg+tofloat(np.stack([r['obs']['matched_prior'] for r in rr]))
                    if policy.use_action_prior:
                        lg=lg+C.shared_prior_entropy*ft[:,:,1]+C.shared_prior_difficulty*ft[:,:,3]
                    d=dist_mask(lg,qm);lp=d.log_prob(selected_q);entropy=d.entropy()
                else:
                    a=policy.actor
                    selected_plan=torch.tensor([r['actions']['plan'] for r in rr],device=DEVICE,dtype=torch.long)
                    selected_kc=torch.tensor([r['actions']['kc'] for r in rr],device=DEVICE,dtype=torch.long)
                    mask_f=qm.float()
                    cov=tofloat(np.stack([r['obs']['covered'] for r in rr]).astype(np.float32))
                    den=mask_f.sum(1).clamp_min(1)
                    summary=torch.stack([(ft[:,:,2]*mask_f).sum(1)/den,
                        (ft[:,:,1]*mask_f).sum(1)/den,cov.mean(1),mask_f.mean(1)],dim=1)
                    if policy.ablation=='no_planning':
                        log0=torch.zeros(len(rr),device=DEVICE);ent0=log0
                    else:
                        d0=torch.distributions.Categorical(logits=a.plan(torch.cat([st,summary],dim=1)))
                        log0=d0.log_prob(selected_plan);ent0=d0.entropy()
                    ohp=F.one_hot(selected_plan,3).float()
                    kc_logit=a.kc(torch.cat([st,ohp,summary],dim=1))
                    kc_unc=((ft[:,:,1]*mask_f)@qkc_t)/((mask_f@qkc_t).clamp_min(1))
                    novelty=1-cov
                    strength=1.0
                    wnew=torch.where(selected_plan==0,.85,torch.where(selected_plan==1,0.0,.44))
                    wunc=torch.where(selected_plan==0,0.0,torch.where(selected_plan==1,.85,.40))
                    if not getattr(a,'no_kc_prior',False):
                        kc_logit=kc_logit+strength*(wnew[:,None]*novelty+wunc[:,None]*kc_unc)
                    d1=dist_mask(kc_logit,km);log1=d1.log_prob(selected_kc);ent1=d1.entropy()
                    match=qkc_t[:,selected_kc].T
                    features=torch.cat([ft,match[:,:,None],(ft[:,:,1]*match)[:,:,None],(ft[:,:,2]*match)[:,:,None]],dim=2)
                    xx=(a.state_proj(st)[:,None,:]+a.feature_proj(features)
                        +a.item_embeddings.weight[None,:,:]
                        +a.plan_embeddings(selected_plan)[:,None,:]
                        +a.kc_embeddings(selected_kc)[:,None,:])
                    lg=a.score(xx).squeeze(-1)
                    if not getattr(a,'no_semantic_prior',False):
                        lg=lg+.65*features[:,:,6]+.18*features[:,:,5]-.25*ft[:,:,1]
                    if policy.use_action_prior:
                        lg=lg+C.shared_prior_entropy*ft[:,:,1]+C.shared_prior_difficulty*ft[:,:,3]
                    d2=dist_mask(lg,qm&(match>0))
                    lp=log0+log1+d2.log_prob(selected_q)
                    entropy=ent0+ent1+d2.entropy()
                ratio=torch.exp(lp-oldlp)
                loss_policy=-torch.minimum(ratio*av,torch.clamp(ratio,1-C.clip,1+C.clip)*av).mean()
                value=policy.critic(gl).squeeze(-1)
                loss=loss_policy+.5*F.mse_loss(value,ret)-.001*entropy.mean()
                optimizer.zero_grad();loss.backward();nn.utils.clip_grad_norm_(policy.parameters(),.5)
                optimizer.step();losses.append(float(loss.detach()))
        return float(np.mean(losses))

## 6.3 Resume aman setiap update

File PPO progres disimpan setelah setiap update. Ketika Colab disconnect, notebook membaca optimizer, model, histori, dan nomor update. SHA dataset, split, DKT, serta konfigurasi dicocokkan sebelum melanjutkan.

In [ ]:
# Kode eksperimen asli; dilewati pada MODE='report' tanpa training.
if MODE not in ('report', 'public'):
    POLICIES={};HISTORIES={}

    def train_policy(kind,seed=42,ablation='none'):
        if MODE not in ('smoke','full','multiseed','ablation','kc_smoke','kc_audit'):
            raise RuntimeError('Training tidak boleh dijalankan saat freeze/final')
        seed_everything(seed)
        old_cov,old_info,old_rank=C.reward_coverage,C.reward_information,C.reward_rank
        if ablation=='no_coverage_reward':C.reward_coverage=0.0
        if ablation=='no_information_reward':C.reward_information=0.0
        if ablation=='no_rank_reward':C.reward_rank=0.0
        try:
            model=Policy(kind,ablation).to(DEVICE)
            optimizer=torch.optim.Adam(model.parameters(),lr=C.ppo_lr)
            label=kind if ablation=='none' else f'{kind}_{ablation}'
            run_dir=MODE_ROOT/f'seed{seed}'
            (run_dir/'ppo').mkdir(parents=True,exist_ok=True)
            ckptdir=run_dir/'ppo'
            progress=ckptdir/f'{label}_progress.pt'
            best_path=ckptdir/f'{label}_best.pt'
            history_path=run_dir/f'{label}_history.csv'
            training_id={ 'mode':MODE,'kind':kind,'ablation':ablation,'seed':seed,
              'data_sha256':DATASET_SHA,'split_sha256':SPLIT_SHA,'dkt_sha256':sha256(DKT_BEST),
              'difficulty_sha256':DIFFICULTY_SHA,'config':asdict(C)}
            training_sha=hashlib.sha256(json.dumps(training_id,sort_keys=True).encode()).hexdigest()
            best=-float('inf');start=1;log=[]
            if progress.exists():
                cp=torch.load(progress,map_location=DEVICE,weights_only=True)
                if cp['signature']!=training_sha:
                    raise RuntimeError(f'Checkpoint {label} tidak cocok identitas eksperimen. Jangan timpa file.')
                model.load_state_dict(cp['model']);optimizer.load_state_dict(cp['optimizer'])
                log=cp['history'];best=float(cp['best']);start=int(cp['update'])+1
                print(f'RESUME {label} seed={seed} dari update {start}')
            for upd in range(start,C.updates+1):
                seed_everything(seed+upd*1237)
                rr=np.random.default_rng(seed+upd*1139)
                model.train();transitions=[]
                cases=get_cases('train')
                sampled=rr.choice(len(cases),size=C.episodes_update,replace=len(cases)<C.episodes_update)
                for idx in sampled:transitions.extend(run_episode(model,cases[int(idx)]))
                loss=(ppo_update(model,optimizer,transitions) if C.smoke
                      else vectorized_ppo_update(model,optimizer,transitions))
                validate=C.smoke or upd==1 or upd%C.validation_interval_full==0 or upd==C.updates
                if validate:
                    score=summarize(evaluate(model,get_cases('val')))
                    u=val_utility(score)
                    log.append({'update':upd,'loss':loss,'utility_val':u,**score})
                    if u>best:
                        best=u;save_torch(best_path,model.state_dict())
                    print(f'{label} seed{seed} update {upd}/{C.updates} KC={score["kc_available"]:.4f} AUC={score["auc"]:.4f} BCE={score["bce"]:.4f} utility={u:.4f}',flush=True)
                else: log.append({'update':upd,'loss':loss,'utility_val':float('nan')})
                save_torch(progress,{'signature':training_sha,'model':model.state_dict(),
                    'optimizer':optimizer.state_dict(),'history':log,'best':best,'update':upd})
                pd.DataFrame(log).to_csv(history_path,index=False)
            if not best_path.exists():raise RuntimeError('Checkpoint policy terbaik tidak ada: '+str(best_path))
            model.load_state_dict(torch.load(best_path,map_location=DEVICE,weights_only=True))
            model.eval()
            return model,pd.DataFrame(log)
        finally:
            C.reward_coverage=old_cov;C.reward_information=old_info;C.reward_rank=old_rank

# Audit pembanding dengan informasi KC setara

Mode `kc_smoke` memeriksa alur dengan anggaran pelatihan kecil. Mode `kc_audit` melatih kebijakan pada lima seed dan mengevaluasi kelompok policy audit. Keduanya membaca ZIP dari `MyDrive/Riset_MARL/input/` dan menulis ke `MyDrive/Riset_MARL_Rerun/<RUN_ID>/`.

`rl_kc` memilih soal secara langsung. Pembanding ini memakai skor KC dari strategi Balanced, dirata-ratakan pada KC yang terkait dengan setiap soal, serta bias item dan prior bersama. Ini menyetarakan sumber informasi dan komponen bias. Besar efek prior dapat berbeda karena MARL menormalkan peluang pada tiap tahap dan RL-KC menormalkan peluang langsung pada seluruh soal layak.

Hasil audit lama tetap menjadi acuan historis. Mode baru mengevaluasi kebijakan hasil pelatihan ulang pada split audit yang sama.

In [ ]:
if MODE in ('kc_smoke', 'kc_audit'):
    class GreedyKCCoverage:
        def eval(self):
            return self

        def choice(self, obs, greedy=True):
            eligible = np.flatnonzero(obs['mask_q'])
            gains = qkc[eligible][:, ~obs['covered']].sum(axis=1)
            question = int(eligible[np.argmax(gains)])  # Seri: ID soal terkecil.
            return {'q': question}, None, None

    audit_cases = get_cases('audit')
    if C.smoke:
        audit_cases = audit_cases[:20]
    seeds = [SEED] if C.smoke else MULTI_SEEDS
    greedy_rows = evaluate(GreedyKCCoverage(), audit_cases)
    records = []
    jobs = [('rl_flat', 'none'), ('rl_feat', 'none'), ('rl_kc', 'none'),
            ('marl', 'none'), ('marl', 'no_planning'), ('marl', 'no_special_priors')]
    for seed in seeds:
        records.append({'seed': seed, 'model': 'greedy_kc', **summarize(greedy_rows)})
        for kind, ablation in jobs:
            policy, history = train_policy(kind, seed, ablation)
            rows = evaluate(policy, audit_cases)
            assert [r['sid'] for r in rows] == [r['sid'] for r in greedy_rows]
            assert all(np.array_equal(a['y'], b['y']) for a, b in zip(rows, greedy_rows))
            label = kind if ablation == 'none' else f'{kind}_{ablation}'
            records.append({'seed': seed, 'model': label, **summarize(rows)})
            pd.DataFrame(records).to_csv(TABLES / 'kc_audit_progress.csv', index=False)

    audit = pd.DataFrame(records)
    audit.to_csv(TABLES / 'kc_audit_by_seed.csv', index=False)
    metrics = ['kc_available', 'kc_global', 'auc', 'bce']
    summary = audit.groupby('model')[metrics].agg(['mean', 'std'])
    summary.columns = ['_'.join(column) for column in summary.columns]
    summary.to_csv(TABLES / 'kc_audit_summary.csv')
    primary = audit.set_index(['seed', 'model'])[metrics].unstack('model')
    differences = pd.DataFrame({
        metric: primary[metric]['marl'] - primary[metric]['rl_kc']
        for metric in metrics
    })
    differences.to_csv(TABLES / 'marl_minus_rl_kc_by_seed.csv')
    display(audit.round(5), summary.round(5), differences.round(5))


In [ ]:
import json

publishable_tables = (
    'kc_audit_by_seed.csv',
    'kc_audit_summary.csv',
    'marl_minus_rl_kc_by_seed.csv',
)
archive_path = MODE_ROOT / f'{RUN_ID}_{MODE}_github_results.zip'
temporary_path = archive_path.with_suffix('.zip.partial')
with zipfile.ZipFile(temporary_path, mode='w', compression=zipfile.ZIP_DEFLATED) as archive:
    for filename in publishable_tables:
        archive.write(TABLES / filename, arcname=filename)
    metadata = {
        'run_id': RUN_ID,
        'mode': MODE,
        'seeds': seeds,
        'test_length': C.test_length,
        'audit_students': len(audit_cases),
        'dataset_sha256': sha256(dataset_zip),
        'evaluation': 'historical policy audit',
    }
    archive.writestr('run_manifest.json', json.dumps(metadata, indent=2))
temporary_path.replace(archive_path)
print('Arsip hasil agregat untuk GitHub:', archive_path)
print(f'Direktori GitHub: results/fresh-rerun/{RUN_ID}/{MODE}/')
